### 1. Подготовка датасета

Целью тренировки нейросети является создание переводчика с нейтрального русского языка на более токсичный вариант.<br>

За основу был взят датасет [Toxic Russian Comments](https://www.kaggle.com/datasets/alexandersemiletov/toxic-russian-comments) c Kaggle. Он содержит комментарии оставленные пользователями в соцсети **ok.ru**, они помечены такими метками как:<br>
 - NORMAL - содержит нейтральные комментарии<br>
 - INSULT - содержит уничижительные комментарии<br>
 - THREAT - содержит комментарии с угрозами<br>
 - OBSCENITY - содержит комментарии с непристойностями/нецензурной лексикой<br>

Для формирования датасета были выбраны все комментарии кроме нейтральных, длинной не более 30 слов (знаки препинания учитывались как слова). Далее каждый комментарий прошел через инференс LLM [google/gemma-4-31B](https://huggingface.co/google/gemma-4-31B) с системным промптом указывающим перевести комментарий в более нейтральную форму. В итоге получился сырой датасет в форме<br>
<br>токсичный комментарий **\t** нейтральный комментарий<br><br>
   Стоит отметить, что использование LLM для автоматической разметки привело к существенному зашумлению итогового датасета вследствии следующих проблем - галлюцинаций нейросети, неоптимального системного промпта, неоптимального выбора самой нейросети для инференса.<br>
Как следствие данный ноутбук иллюстрирует принцип GARBAGE IN - GARBAGE OUT.

In [1]:
from io import open
import unicodedata
import string
import re
import random
import time
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import warnings

Изначально общий словарь датасета достигал около 73 000 токенов, что избыточно много для обучения модели. Чтобы значительно уменьшить размер словарей и очистить данные от опечаток и редких выбросов, была проведена жесткая фильтрация: из датасета были полностью удалены все пары фраз, в которых встречается хотя бы одно слово с частотностью менее 3 раз за весь корпус текстов, что также негативно повлияло на размер датасета.

Очистим фразы от лишних символов, оставляя только буквы и сформируем пары нейтральный/токсичный комментарий:

In [ ]:
import re

def clean_text(text):
    text = text.lower().strip()
    # Оставляем только кириллицу и знаки препинания
    text = re.sub(r"[^а-яА-ЯёЁ]+", r" ", text)
    # Убираем лишние пробелы
    return " ".join(text.split()).strip()

pairs = []
with open('toxic_dataset.tsv', 'r', encoding='utf-8') as f:
    for line in f:
        if '\t' in line:
            parts = line.split('\t')
            if len(parts) == 2:
                # порядок фраз в датасете "токсичная/нейтральная" поэтому сначала берем вторую часть строки
                pairs.append([clean_text(parts[1]), clean_text(parts[0])])

print(f"Загружено пар: {len(pairs)}")

### Создаем класс словаря:

In [3]:
SOS_token = 0
EOS_token = 1
UNK_token = 2
PAD_token = 3

class LanguageVocabulary:
    def __init__(self, name):
        self.name = name
        self.word2index = {"SOS": 0, "EOS": 1, "UNK": 2, "PAD": 3}
        self.word2count = {"SOS": 0, "EOS": 0, "UNK": 0, "PAD": 0}
        self.index2word = {0: "SOS", 1: "EOS", 2: "UNK", 3: "PAD"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

### Заполняем словари нормализованными данными

In [4]:
# --- Ячейка 2: Строим словари напрямую (без rubert, без UNK замен) ---
input_lang = LanguageVocabulary('neutral')
output_lang = LanguageVocabulary('toxic')
for pair in pairs:
    input_lang.add_sentence(pair[0])
    output_lang.add_sentence(pair[1])
print(f"Размер входного словаря: {input_lang.n_words}")
print(f"Размер выходного словаря: {output_lang.n_words}")

Размер входного словаря: 8509
Размер выходного словаря: 10051


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Создаем функции для кодирования и создания батчей для передачи в модель:

In [6]:
def indexes_from_sentence(lang, sentence):
    return [lang.word2index.get(word, UNK_token) for word in sentence.split()] + [EOS_token]

def create_batch(batch_pairs):
    # Конвертируем пары в индексы
    input_seqs = [indexes_from_sentence(input_lang, p[0]) for p in batch_pairs]
    target_seqs = [indexes_from_sentence(output_lang, p[1]) for p in batch_pairs]

    # Сортируем по убыванию длины входа (нужно для pack_padded_sequence)
    paired = sorted(zip(input_seqs, target_seqs), key=lambda x: len(x[0]), reverse=True)
    input_seqs, target_seqs = zip(*paired)

    input_lengths = [len(s) for s in input_seqs]
    target_lengths = [len(s) for s in target_seqs]

    # Паддим до длины самой длинной последовательности в батче
    input_padded = pad_sequence(
        [torch.tensor(s, dtype=torch.long) for s in input_seqs],
        batch_first=False, padding_value=PAD_token
    )  # (max_input_len, batch_size)

    target_padded = pad_sequence(
        [torch.tensor(s, dtype=torch.long) for s in target_seqs],
        batch_first=False, padding_value=PAD_token
    )  # (max_target_len, batch_size)

    return (
        input_padded.to(device),
        target_padded.to(device),
        input_lengths,
        target_lengths
    )


### Создаем классы для энкодера и декодера:

In [7]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.3):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, hidden_size, padding_idx=PAD_token)
        self.gru = nn.GRU(hidden_size, hidden_size,
                          num_layers=num_layers, dropout=dropout)

    def forward(self, input_seqs, input_lengths, hidden=None):
        
        embedded = self.embedding(input_seqs)

        packed = pack_padded_sequence(embedded, input_lengths, enforce_sorted=True)
        outputs, hidden = self.gru(packed, hidden)
        outputs, _ = pad_packed_sequence(outputs)

        return outputs, hidden

    def initHidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)


In [8]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, num_layers=2, dropout=0.3):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_size, hidden_size, padding_idx=PAD_token)
        self.gru = nn.GRU(hidden_size, hidden_size,
                          num_layers=num_layers, dropout=dropout)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):

        embedded = self.embedding(input).unsqueeze(0)  # (1, batch, hidden)
        output = F.relu(embedded)
        output, hidden = self.gru(output, hidden)
        output = F.log_softmax(self.out(output.squeeze(0)), dim=1)  # (batch, vocab_size)
        return output, hidden


### Создадим функцию обучения для работы с батчами:

In [9]:
def train_batch(input_padded, target_padded, input_lengths, target_lengths,
                encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    batch_size = input_padded.size(1)

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    encoder_hidden = encoder.initHidden(batch_size)
    encoder_outputs, encoder_hidden = encoder(input_padded, input_lengths, encoder_hidden)

    decoder_input = torch.full((batch_size,), SOS_token, dtype=torch.long, device=device)
    decoder_hidden = encoder_hidden

    max_target_len = target_padded.size(0)
    loss = 0

    use_teacher_forcing = random.random() < 0.5

    if use_teacher_forcing:
        for di in range(max_target_len):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_padded[di])
            decoder_input = target_padded[di]  # Подаем правильный токен
    else:
        for di in range(max_target_len):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_padded[di])
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze(1).detach()  # Подаем предсказанный токен

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / max_target_len


### Функция которая будет формировать батчи из пар и использовать эти батчи для обучения сети:

In [10]:
def trainIters(encoder, decoder, epochs=10, batch_size=64,
               print_every=50, learning_rate=0.001):
    encoder.train()
    decoder.train()

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss(ignore_index=PAD_token)  # Игнорируем PAD в loss

    for epoch in range(1, epochs + 1):
        random.shuffle(pairs)
        epoch_loss = 0
        n_batches = 0

        start = time.time()

        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i : i + batch_size]
            if len(batch_pairs) < 2:  # Слишком маленький батч для dropout
                continue

            input_padded, target_padded, input_lengths, target_lengths = create_batch(batch_pairs)

            loss = train_batch(
                input_padded, target_padded, input_lengths, target_lengths,
                encoder, decoder,
                encoder_optimizer, decoder_optimizer, criterion
            )
            epoch_loss += loss
            n_batches += 1

            if n_batches % print_every == 0:
                avg = epoch_loss / n_batches
                elapsed = time.time() - start
                print(f'Эпоха {epoch} | Батч {n_batches}/{len(pairs)//batch_size} '
                      f'| Loss: {avg:.4f} | Время: {elapsed:.0f}с')

        avg_epoch_loss = epoch_loss / n_batches
        elapsed = time.time() - start
        print(f'--- Эпоха {epoch} завершена | Avg Loss: {avg_epoch_loss:.4f} '
              f'| Время: {elapsed:.0f}с ---')


### Функция для инференса:

In [11]:
def evaluate(encoder, decoder, sentence, max_length=30):
    with torch.no_grad():
        # Подготавливаем вход как батч из одного элемента
        input_indexes = indexes_from_sentence(input_lang, sentence)
        input_tensor = torch.tensor(input_indexes, dtype=torch.long, device=device).unsqueeze(1)
        input_lengths = [len(input_indexes)]

        encoder_hidden = encoder.initHidden(1)
        encoder_outputs, encoder_hidden = encoder(input_tensor, input_lengths, encoder_hidden)

        decoder_input = torch.tensor([SOS_token], dtype=torch.long, device=device)
        decoder_hidden = encoder_hidden

        decoded_words = []
        for di in range(max_length):
            output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = output.data.topk(1)

            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word.get(topi.item(), 'UNK'))

            decoder_input = topi.squeeze(1).detach()

        return ' '.join(decoded_words)


### Этап обучения:

In [17]:
hidden_size = 256
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print(f'Encoder параметров: {sum(p.numel() for p in encoder.parameters()):,}')
print(f'Decoder параметров: {sum(p.numel() for p in decoder.parameters()):,}')

Encoder параметров: 2,967,808
Decoder параметров: 5,945,667


In [19]:
trainIters(encoder, decoder, epochs=30, batch_size=32,
           print_every=50, learning_rate=0.0003)

Эпоха 1 | Батч 50/522 | Loss: 2.8951 | Время: 1с
Эпоха 1 | Батч 100/522 | Loss: 2.9512 | Время: 1с
Эпоха 1 | Батч 150/522 | Loss: 2.9187 | Время: 1с
Эпоха 1 | Батч 200/522 | Loss: 2.9082 | Время: 2с
Эпоха 1 | Батч 250/522 | Loss: 2.9196 | Время: 2с
Эпоха 1 | Батч 300/522 | Loss: 2.9311 | Время: 3с
Эпоха 1 | Батч 350/522 | Loss: 2.9356 | Время: 3с
Эпоха 1 | Батч 400/522 | Loss: 2.9280 | Время: 4с
Эпоха 1 | Батч 450/522 | Loss: 2.9221 | Время: 4с
Эпоха 1 | Батч 500/522 | Loss: 2.9276 | Время: 5с
--- Эпоха 1 завершена | Avg Loss: 2.9226 | Время: 5с ---
Эпоха 2 | Батч 50/522 | Loss: 2.7518 | Время: 0с
Эпоха 2 | Батч 100/522 | Loss: 2.6737 | Время: 1с
Эпоха 2 | Батч 150/522 | Loss: 2.6828 | Время: 1с
Эпоха 2 | Батч 200/522 | Loss: 2.7436 | Время: 2с
Эпоха 2 | Батч 250/522 | Loss: 2.7552 | Время: 2с
Эпоха 2 | Батч 300/522 | Loss: 2.7706 | Время: 3с
Эпоха 2 | Батч 350/522 | Loss: 2.7572 | Время: 3с
Эпоха 2 | Батч 400/522 | Loss: 2.7451 | Время: 4с
Эпоха 2 | Батч 450/522 | Loss: 2.7530 | Время

Сохраняем полученные веса:

In [26]:
torch.save(encoder.state_dict(), 'toxic_encoder.pth')
torch.save(decoder.state_dict(), 'toxic_decoder.pth')

### Инференс:

In [25]:
encoder.eval()
decoder.eval()
output_sentence = evaluate(encoder, decoder, 'это полная бессмыслица')
print(output_sentence)

чепуха ёбаный <EOS>


### Код для загрузки предобученной модели:

In [14]:
import os
import torch

encoder_path = 'toxic_encoder.pth'
decoder_path = 'toxic_decoder.pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if os.path.exists(encoder_path) and os.path.exists(decoder_path):

    hidden_size = 256
    encoder = EncoderRNN(input_lang.n_words, hidden_size, num_layers=2).to(device)
    decoder = DecoderRNN(hidden_size, output_lang.n_words, num_layers=2).to(device)
    
    encoder.load_state_dict(torch.load(encoder_path, map_location=device))
    decoder.load_state_dict(torch.load(decoder_path, map_location=device))
    
    encoder.eval()
    decoder.eval()

### Проверка качества с помощью метрики BLEU
Метрика BLEU не применялась, так как при данном качестве датасета количественная оценка не имеет смысла.